# Topics of Practice: Data Bigger Than Your Laptop

**GSB 5544 · Computing and Machine Learning for Business Analytics**

This notebook walks through the main functions from the pre-class reading, one short block at a time. Each block names the section of the reading it draws on. Replace every `____` with the right code and run the cell.

**Run this notebook inside your SageMaker Studio JupyterLab space** (reading, §4), not on your laptop. Every code cell begins with a `# RUNS ON:` line that says where it executes. Read it before you press Shift-Enter; the point of today is to notice where the work happens.

Bring the five laptop numbers you wrote down from the reading, §1. Block A needs them.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
%pip install -q s3fs             # harmless if already present

import io
import time
import psutil
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "noaa-ghcn-pds"
REGION = "us-east-1"             # the bucket's region; this instance is in it too

## A. Two machines, side by side

*From the reading, §1 and §4.* You ran the `psutil` cell on your laptop before class. Now run it again: this kernel lives on a rented `ml.t3.medium` in `us-east-1`, and your laptop is a browser tab.

Fill in the call that reports memory and the attribute that gives the memory available **right now**.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
vm = psutil.____()
du = psutil.disk_usage("/")

sagemaker = {
    "physical cores":     psutil.cpu_count(logical=False),
    "logical cores":      psutil.cpu_count(logical=True),
    "total RAM (GB)":     round(vm.total / 1e9, 1),
    "available RAM (GB)": round(vm.____ / 1e9, 1),
    "free disk (GB)":     round(du.free / 1e9, 1),
}
MY_RAM = vm.____          # available RAM of the machine running THIS cell; we compare file sizes against it below
sagemaker

Now paste your laptop's numbers into the dictionary and build the comparison. Fill in the dictionary that holds this machine's numbers.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
laptop = {                        # <- replace every value with the number you wrote down from the reading, §1
    "physical cores":     0,
    "logical cores":      0,
    "total RAM (GB)":     0.0,
    "available RAM (GB)": 0.0,
    "free disk (GB)":     0.0,
}

compare = pd.DataFrame({"laptop": laptop, "SageMaker ml.t3.medium": ____})
compare["SageMaker ÷ laptop"] = (compare["SageMaker ml.t3.medium"] / compare["laptop"]).round(2)
compare

**Check yourself.** The rented machine is probably *smaller* than your laptop. So why rent it at all? Answer in one sentence, as a comment.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# ____

## B. The units of data

*From the reading, §2.* Byte counts are unreadable past a few million. Write the helper that turns them into KB / MB / GB and so on, using base 1000.

Fill in the list of units in ascending order and the line that steps `n` down one unit.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
def human(n, base=1000):
    for unit in [____]:
        if n < base:
            return f"{n:,.1f} {unit}"
        n ____
    return f"{n:,.1f} EB"

print(human(1_500_000))          # expect 1.5 MB
print(human(1_336_457_184))      # expect 1.3 GB
print(human(MY_RAM))             # this machine's available RAM, readable

**Check yourself.** A "1 TB" drive shows as 931 GB in your file browser. Which base does the drive maker use, and which does the operating system use? Write your answer in the cell below as a comment.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# Drive maker uses base ____; the OS uses base ____.
# 1e12 / 1024**3 = ____ GiB, which the OS labels "GB".

## C. The vocabulary of AWS

*From the reading, §3.* Fill in the term that each definition describes. The cell checks your answers.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
vocab = {
    "A named container for objects; its name is globally unique":            "____",
    "One stored item plus its metadata such as size":                         "____",
    "The full name of an object inside its bucket":                           "____",
    "The start of a key, used to list everything 'under' something":          "____",
    "A geographic cluster of data centers such as us-east-1":                 "____",
    "A rented virtual computer":                                              "____",
    "The size of a rented computer: cores, RAM, and a price per hour":        "____",
    "A SQL engine that scans files in S3 across many machines, billed per byte": "____",
    "The service that decides who may do what":                               "____",
}

expected = {"bucket", "object", "key", "prefix", "region", "ec2", "instance type", "athena", "iam"}
given = {v.strip().lower() for v in vocab.values()}
print("All correct" if given == expected else f"Check: {expected - given}")

## D. Connect without an account

*From the reading, §5.* Public datasets on S3 can be read with an **unsigned** request. Build the client, then list the top-level "folders" of the GHCN bucket.

This machine *has* AWS credentials (it is inside your lab account), but an unsigned request does not use them. The public bucket does not care who is asking.

Fill in the signature setting that makes the request anonymous, and the delimiter that makes S3 group keys like folders.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
s3 = boto3.client(
    "s3",
    region_name=REGION,
    config=Config(signature_version=____),
)

resp = s3.list_objects_v2(Bucket=BUCKET, Delimiter=____)

print("Prefixes :", [p["Prefix"] for p in resp.get("____", [])])
print("Files    :", [o["Key"] for o in resp.get("Contents", [])][:6])

Look at the output. `csv/` is not a folder. In one sentence, what is it? Type your answer as a comment.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# ____

## E. Read a size without reading the file

*From the reading, §5.* Every object carries its size as metadata. List the by-year files for the 2020s and print each size in readable units.

Fill in the prefix that selects `csv/by_year/2020.csv` through `2026.csv`, and the field that holds an object's size.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=____)

sizes = {}
for obj in resp["Contents"]:
    sizes[obj["Key"]] = obj[____]
    print(f"{obj['Key']:<28} {human(obj[____])}")

BY_YEAR_2024_BYTES = sizes["csv/by_year/2024.csv"]     # keep; block J compares against it

Now the comparison that the whole reading builds toward. pandas typically needs about **3×** a text CSV's size in memory. For each file, decide whether a plain `pd.read_csv` would fit in **this machine's** available RAM. (Renting a machine did not change the arithmetic.)

Fill in the multiplier and the comparison.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
MULT = ____

for key, size in sizes.items():
    need = size * MULT
    fits = need ____ MY_RAM
    print(f"{key:<28} needs ~{human(need):>9}   fits on this ml.t3.medium: {fits}")

## F. Peek at a file with a range request

*From the reading, §5.* You can read the first few hundred bytes of a gigabyte file to see its shape. Do that for the 2024 by-year file and check whether it has a header row. Never assume either way.

Fill in the `Range` header for the first 300 bytes.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
key = "csv/by_year/2024.csv"

obj = s3.get_object(Bucket=BUCKET, Key=key, Range=____)
first = obj["Body"].read().decode()
print(first)

has_header = first.upper().startswith(____)
print("Header row present:", has_header)

## G. Extract one station, not one year

*From the reading, §6.* Lever 1 is picking the right object. The Cal Poly station (`USC00047851`, SAN LUIS OBISPO POLY) has records since 1893 in a single file of about 7 MB. Read it, keeping only the four columns you need. The bytes travel from S3 to this instance over AWS's own network; nothing reaches your laptop except the printed output.

Fill in the station key, the header logic, and the `usecols` list.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
COLS = ["id", "date", "element", "value", "m_flag", "q_flag", "s_flag", "obs_time"]
STATION = "USC00047851"
key = f"csv/by_station/{____}.csv"

# size first, always
print("Size:", human(s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"]))

# header check
first = s3.get_object(Bucket=BUCKET, Key=key, Range="bytes=0-99")["Body"].read().decode()
has_header = first.upper().startswith("ID,")

raw = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
df = pd.read_csv(
    io.BytesIO(raw),
    header=____,
    names=COLS,
    usecols=____,
    dtype={"id": "string", "element": "string"},
)

print(df.shape)
print("In memory:", human(df.memory_usage(deep=____).sum()))
df.head()

## H. Transform

*From the reading, §6, "The units trap".* Dates are stored as `YYYYMMDD` integers and temperatures in tenths of a degree Celsius. Fix both, keep only daily maximum temperature, and compute a yearly mean.

Fill in the date format string, the element code, the unit conversion, and the resampling rule for year-end.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
df["date"] = pd.to_datetime(df["date"].astype(str), format=____)

tmax = df[df["element"] == ____].copy()
tmax["tmax_c"] = tmax["value"] ____

yearly = (
    tmax.set_index("date")["tmax_c"]
        .resample(____)          # "Y" on pandas < 2.2
        .mean()
)

print(f"{len(tmax):,} TMAX readings, {yearly.index.min().year}–{yearly.index.max().year}")
yearly.tail()

## I. Bring the visual

*From the reading, §7.* One picture that came from a bucket you never downloaded, drawn on a machine you do not own.

Fill in the series to plot and the y-axis label with its unit.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(yearly.index, ____, linewidth=1.2)
ax.set_ylabel(____)
ax.set_title(f"{STATION}: yearly mean of daily maximum temperature")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## J. Ask Athena for the same station

*From the reading, §6, "the fourth lever".* So far every filter has run on the machine holding this kernel. Athena runs the filter on AWS's fleet, against the Parquet copy of the data, and hands back only the matching rows. Three cells: a results bucket you own, a table definition, and the query.

**Results bucket.** Athena writes every result to S3, so you need a bucket of your own. These clients are *signed*: they act as you, because this bucket is yours, not public. Fill in the bucket name in the create call.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
sts     = boto3.client("sts",    region_name=REGION)
athena  = boto3.client("athena", region_name=REGION)
s3_mine = boto3.client("s3",     region_name=REGION)

ACCOUNT        = sts.get_caller_identity()["Account"]
RESULTS_BUCKET = f"gsb5544-athena-{ACCOUNT}"     # bucket names are global, so include your account id
RESULTS        = f"s3://{RESULTS_BUCKET}/athena/"

s3_mine.create_bucket(Bucket=____)   # in us-east-1 no LocationConstraint is needed
print("Results will land in", RESULTS)

**A helper and a table.** `run_athena` starts a query, waits for it, and returns the query id plus Athena's statistics, which include the bytes scanned. The table definition tells Athena where the Parquet files are and how the `YEAR=`/`ELEMENT=` key layout maps to columns; nothing is copied, and DDL statements are free. Note the backticks around `date`: it is a reserved word in SQL.

Fill in the `boto3` call that starts a query.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
ATHENA_BYTES = 0                                  # running total of everything scanned today; block K audits it

def run_athena(sql, database=None):
    """Start a query, wait for it, return (query id, statistics)."""
    global ATHENA_BYTES
    kwargs = {"QueryString": sql, "ResultConfiguration": {"OutputLocation": RESULTS}}
    if database:
        kwargs["QueryExecutionContext"] = {"Database": database}
    qid = athena.____(**kwargs)["QueryExecutionId"]
    while True:
        q = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        state = q["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1)
    if state != "SUCCEEDED":
        raise RuntimeError(q["Status"].get("StateChangeReason", state))
    ATHENA_BYTES += q["Statistics"].get("DataScannedInBytes", 0)
    return qid, q["Statistics"]


def athena_df(qid):
    """Read a finished query's result CSV from the results bucket."""
    obj = s3_mine.get_object(Bucket=RESULTS_BUCKET, Key=f"athena/{qid}.csv")
    return pd.read_csv(io.BytesIO(obj["Body"].read()))

In [ ]:
# RUNS ON: Athena fleet (results land in S3)
run_athena("CREATE DATABASE IF NOT EXISTS gsb5544")

DDL = '''
CREATE EXTERNAL TABLE IF NOT EXISTS gsb5544.ghcn (
    id         string,
    `date`     string,
    data_value bigint,
    m_flag     string,
    q_flag     string,
    s_flag     string,
    obs_time   string
)
PARTITIONED BY (year int, element string)
STORED AS PARQUET
LOCATION 's3://noaa-ghcn-pds/parquet/by_year/'
TBLPROPERTIES (
    'projection.enabled'        = 'true',
    'projection.year.type'      = 'integer',
    'projection.year.range'     = '1750,2030',
    'projection.element.type'   = 'enum',
    'projection.element.values' = 'TMAX,TMIN,PRCP,SNOW,SNWD,TAVG',
    'storage.location.template' = 's3://noaa-ghcn-pds/parquet/by_year/YEAR=${year}/ELEMENT=${element}/'
)
'''
run_athena(DDL, database="gsb5544")
print("Table gsb5544.ghcn is defined. Bytes scanned so far:", human(ATHENA_BYTES))

**The query.** Ask for the same station you pulled in block G, but only 2024 and only `TMAX` and `PRCP`. The `WHERE` clause is the extraction. Fill in the year and the name of the statistic that reports bytes scanned.

In [ ]:
# RUNS ON: Athena fleet (results land in S3)
SQL = f'''
SELECT id, "date", data_value
FROM   gsb5544.ghcn
WHERE  year = ____
  AND  element IN ('TMAX', 'PRCP')
  AND  id = '{STATION}'
'''

qid, stats = run_athena(SQL, database="gsb5544")
scanned = stats["____"]
station_2024 = athena_df(qid)

print(f"Rows returned : {len(station_2024):,}")
print(f"Bytes scanned : {human(scanned)}")
print(f"Query time    : {stats['TotalExecutionTimeInMillis'] / 1000:.1f} s")
print(f"Cost          : ${max(scanned, 10_000_000) / 1e12 * 5:.4f}   ($5 per TB, 10 MB minimum)")
print(f"2024 CSV was  : {human(BY_YEAR_2024_BYTES)}  ({BY_YEAR_2024_BYTES / scanned:.0f}× more bytes than Athena scanned)")
station_2024.head()

Athena scanned a fraction of the 1.3 GB by-year CSV from block E and returned only this station's rows, in seconds. Which lever from the reading did the `YEAR=`/`ELEMENT=` key layout let Athena pull? One sentence, as a comment.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# ____

## K. Stop the meter

*From the reading, §7.* A session on a rented machine ends the same way every time. First, empty the results bucket from here. Fill in the call that lists the bucket and the call that deletes a batch of objects.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
listing = s3_mine.____(Bucket=RESULTS_BUCKET)
objects = listing.get("Contents", [])
print(f"{len(objects)} result objects, {human(sum(o['Size'] for o in objects))}")

if objects:
    s3_mine.____(Bucket=RESULTS_BUCKET, Delete={"Objects": [{"Key": o["Key"]} for o in objects]})

print("Bucket empty :", "Contents" not in s3_mine.list_objects_v2(Bucket=RESULTS_BUCKET))
print("Athena today :", human(ATHENA_BYTES), f"scanned, about ${max(ATHENA_BYTES, 10_000_000) / 1e12 * 5:.4f}")

Second, stop the instance. A cell cannot stop the machine it is running on without killing itself, so this part happens in the Studio window. In this order:

- [ ] File → Save All. Download anything you want to keep.
- [ ] In Studio, open **JupyterLab spaces**, find `gsb5544`, click **Stop**, and wait until the status reads *Stopped*.
- [ ] Read the budget at the top of the Learner Lab page. Today should have cost well under a dollar.
- [ ] Click **End Lab**.

Before you do, write one sentence, as a comment, that answers: *why did reading the station file work when reading the year file would not have, and what did Athena add on top of that?*

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# ____